In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import ROOT as root
from ROOT import TH2F
from ROOT import TH1F
from openpyxl import Workbook
import pytz
from datetime import datetime
import json
import pandas as pd

Welcome to JupyROOT 6.30/04


In [2]:
root.gStyle.SetOptStat(0)
palette = np.array([3,5,2], dtype=np.int32)
  
  
root.gStyle.SetTitleOffset( 1.3, "z" )
root.gStyle.SetLabelOffset( 0., "z" )
#root.gStyle.SetTitleSize(0.06,"z") 
root.gStyle.SetLabelSize(0.035,"z")
root.gStyle.SetPaintTextFormat("1.1f")

   
###################################/ 
## THIS SCRIPT ASSUMES raw voltages in [V] and raw currents in [A] #
## Parameters below are in [V] and [uA]                            #
###################################/


V_current_level = 0.,0.,210.,210.,0.,210.,210.,210.,210.,210.,210.,210.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,210. #Leakage current measured at this V
V_current_monitor = V_current_level #Voltage up to which the current level is monitored
VBD_expected = 0.,0.,230.,250.,0.,250.,250.,250.,250.,250.,250.,250.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,250.
VBD_expected_sigma = 5.,5.,5.,5.,5.,5.,5.,5.,5.,5.,5.,5.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,5.
V_min_kfactor = 100.  # 100 #k-factor not used to calculate VBD if VBD<V_min_kfactor
I_thr = 10000*1E-3   #sensor discarded if I > I_thr [nA] in the voltage operation range [ 0-V_current_monitor ]
I_compliance = 50*1E-3 # VBD calculation begins when I < I_compliance  [nA]
I_compliance_minimum = 50*1E-3 # [nA] VBD calculation performed only if compliance was set above this threshold
k_thr = 20. # 20  k value to define VBD using k-factor method
current_conversion_value = 1E6 # conversion from [A] (raw data) to [nA] (used in the final plots)
start_bd_calculation = 2 #BD calculation start from this sampled bias point: avoid considering the very first voltages of the bias sweep 
low_iv_range = 0.1  #0.1 #low and high ranges for I@100V plot [uA]
high_iv_range = 1000 #10
low_vbd_range = 200 #190 #low and high ranges for VBD plot [V]
high_vbd_range = 270 #230
n_wafer = 28 #
n_row = 6 #
n_col = 6 #
n_pad = 256 # PAD
n_sensor = 24 # SENSOR
start_row = 1 #the first row of the wafer to be measured
start_col = 1 #the first column of the wafer to be measured 
start_wafer = 1 #first wafer number in the dataset
start_pad = 1 #the first row of the wafer to be measured
start_sensor = 1 #the first sensor of the wafer to be measured 
bcurrent = True
bvoltage = False
bcategory = False
bnoisy = False
save = False
verbose = True
write_json = True  #produces the json file needed to upload the tests to the ETL database

dtz = datetime(2025, 3, 28, 12, 0, 0)
dtz = dtz.replace(tzinfo=pytz.utc)
dtz.astimezone(pytz.timezone("Europe/Rome"))

if(bcategory): 
    root.gStyle.SetPalette(3,palette) # custom palette used for Categories

if(bvoltage):
    #root.gStyle.SetPalette("Black Body") # palette for VBD
    root.gStyle.SetPalette(70) # palette for VBD

if(bcurrent or bnoisy):
    #root.gStyle.SetPalette("Black Body") # palette for I@V_current_level
    root.gStyle.SetPalette(70)
    root.TColor.InvertPalette()

invert_polarity=True

#file_qa = root.TFile.Open("../../root_files/HPK5_16x16_IVtree_ETL-site_bis.root")  unumbered files
file_qa = root.TFile.Open("../../root_files/HPK5_16x16_IVtree_ETL-site.root")
tree_qa = file_qa.Get("Tree")
  
counter_qa = 0
I_qa_100V = 0.
VBD_qa_u = 0.
VBD_qa_d = 0.
k_qa_u = []
k_qa_d = []
k_qa_u.append(0.)
k_qa_u.append(0.)
k_qa_d.append(0.)
k_qa_d.append(0.)

tests_json = []

current_levels_counter=[0,0,0,0,0,0]
vcount_vbd_qa = [[0 for j in range(n_sensor)] for k in range(n_wafer)]
vcount_i_qa = [[0 for j in range(n_sensor)] for k in range(n_wafer)]
vcount_bump_qa = [[0 for j in range(n_sensor)] for k in range(n_wafer)]
vvbd_qa = [[0 for j in range(n_sensor)] for k in range(n_wafer)]
vi_qa = [[0 for j in range(n_sensor)] for k in range(n_wafer)]
vbump_qa = [[0 for j in range(n_sensor)] for k in range(n_wafer)]
vi_raw = [[[] for j in range(n_sensor)] for k in range(n_wafer)]
v_raw = [[[] for j in range(n_sensor)] for k in range(n_wafer)]
#vnoisy_qa = None
hI_qa_100V = []
hV_qa = []
hbump_qa = []
hnoisy_qa = []
hcount_qa = []
min_I = []

for i in range(n_wafer):
    hI_qa_100V.append( TH2F( "I_qa_"+str(V_current_level[i])+"V_W"+str(i+1), "I@"+str(V_current_level[i])+"V on-wafer W"+str(i+1), n_col,start_col,(start_col+n_col),n_row,start_row,(start_row+n_row) ) )
    #hV_qa.append( TH2F( "V_qa_W"+str(i+1), "VBD on-wafer W"+str(i+1), n_col,start_col,(start_col+n_col),n_row,start_row,(start_row+n_row) ) )
    hV_qa.append( TH2F( "V_qa_W"+str(i+1), "VBD on-wafer W"+str(i+1), 6,1,7,6,1,7 ) )
#    min_I.append(1000)

    hbump_qa.append( TH2F( "bump_qa_W"+str(i+1), "Categories on-wafer W"+str(i+1), n_col,start_col,(start_col+n_col),n_row,start_row,(start_row+n_row)) )
#    hnoisy_qa.append( TH2F( "noisy_qa_W"+str(i+1), "Bad pads on-wafer W"+str(i+1), n_col,start_col,(start_col+n_col),n_row,start_row,(start_row+n_row)) )

dumb_counter= 0. # Total number of sensors/measurements in the root file
non_empty_counter= 0. # Sensors with both I and V arrays having size !=0 (the sensor was in fact measured)
non_zero_counter= 0. # Sensors with both I and V arrays having values !=0 (the sensor was measured and can be bias
total_sensors_counter_qa= 0. #Fraction of sensors whose category can be properly defined
good_sensors_counter_qa= 0. #Fraction of GOOD sensors
bad_sensors_counter_qa= 0. #Fraction of BAD sensors
medium_sensors_counter_qa= 0. #Fraction of MEDIUM sensors
iv_quality = bool() # Boolean reflecting the quality of the IV curve: sensor is BAD whenever iv_quality is set to False

sensor = "Prototype LGAD"
sensor_geom = "16x16-T9"
vendor = "HPK"
production = "S16694-34190"
user = "fsiviero" # cern user of who's uploading the test
location = "Torino" # where the test was performed

if(write_json):
    json_filename="ETL_site_IV"
    
save_path = '/Users/icosivi/Desktop/plot_HPK5/'
wb = pd.read_excel('/Users/icosivi/Desktop/plot_HPK5/HPK5_16x16_ETLdb.xlsx')

In [3]:
for event in tree_qa:
 if event.sensor<25:
    dumb_counter+=1
    if(int(event.IBACK.size())>=start_bd_calculation and int(event.V.size())>=start_bd_calculation ):
      non_empty_counter+=1

      if( event.IBACK.at(event.IBACK.size()-1)<0 ): 
        Iback = [float(-i) for i in event.IBACK]
        for idx, i in enumerate(event.IBACK):  
          if(idx<=event.V.size()-1):
           vi_raw[event.wafer-start_wafer][event.sensor-start_sensor].append(float(-i))  
        #if(verbose): 
          #print("Sensor from wafer "+str(event.wafer)+" row "+str(event.row)+" sensor "+str(event.sensor)+" with negative current detected. Changing polarity.")
      else:
        Iback = [i for i in event.IBACK]
        for idx, i in enumerate(event.IBACK):  
          if(idx<=event.V.size()-1):
           vi_raw[event.wafer-start_wafer][event.sensor-start_sensor].append(float(i))  

      if( event.V.at(event.V.size()-1)<0 ): 
        Vbias = [-i for i in event.V]  
        for idx, i in enumerate(event.V):  
         if(idx<=event.IBACK.size()-1):
           v_raw[event.wafer-start_wafer][event.sensor-start_sensor].append(float(-i))        
        #if(verbose): 
          #print("Sensor from wafer "+str(event.wafer)+" row "+str(event.row)+" sensor "+str(event.sensor)+" with negative Voltage detected. Changing polarity.")
      else:
        Vbias = [i for i in event.V]
        for idx, i in enumerate(event.V): 
          if(idx<=event.IBACK.size()-1): 
           v_raw[event.wafer-start_wafer][event.sensor-start_sensor].append(float(i))
           

      non_zero_current = sum(Iback) 
      non_zero_voltage = sum(Vbias)


      if(non_zero_voltage!=0 and non_zero_current!=0):
        non_zero_counter+=1
    
        iv_quality = True
        I_qa_100V = -1000
        VBD_qa_u = -1000.
        VBD_qa_d = -1000.
        
        type_counter = 0
        type_counter_I_qa = 0
        type_counter_V_qa_u = 0
        type_counter_V_qa_d = 0
      
      
        ####### I@V_current_level calculation #####
        for i in range(len(Iback)):
          if( i!=0 and Vbias[i-1]<V_current_level[event.wafer-start_wafer] and Vbias[i]>=V_current_level[event.wafer-start_wafer] ):
            if( Vbias[i]==V_current_level[event.wafer-start_wafer] ): 
              I_qa_100V = Iback[i]
            else: 
              I_qa_100V = (((Iback[i]-Iback[i-1]))/(Vbias[i]-Vbias[i-1]))*(V_current_level[event.wafer-start_wafer]-Vbias[i]) + Iback[i]
            break
          	
    
        for i in range(len(Iback)):
          if( Vbias[i]<=V_current_monitor[event.wafer-start_wafer] and Iback[i] > I_thr*(1./current_conversion_value) ):
            iv_quality=False
            break

        I_qa_100V = current_conversion_value*I_qa_100V
        if( I_qa_100V<0. ): 
          iv_quality=False
          
        ### VBD calculation ###
        if( Iback[ -1 ]>I_compliance_minimum*(1./current_conversion_value) ):
          ### VBD "up" (Calculation start from the end of the Voltage array downwards) ###
          for i in reversed(range(start_bd_calculation,int(len(Iback))-1)):
            if(Vbias[len(Iback)-1]<V_min_kfactor): 
                VBD_qa_u = Vbias[len(Iback)-1]
                break 
            
            if( Iback[i]<I_compliance*(1./current_conversion_value) and Vbias[i]>=V_min_kfactor ):
              k_qa_u[0] = ( (Iback[i]-Iback[i-1])/(Vbias[i]-Vbias[i-1]) )*(Vbias[i]/Iback[i]) 
              k_qa_u[1] = ( (Iback[i+1]-Iback[i])/(Vbias[i+1]-Vbias[i]) )*(Vbias[i]/Iback[i]) 
              if( k_qa_u[0]<k_thr and k_qa_u[1]>=k_thr ):
                VBD_qa_u = Vbias[i] 
                break 
              
         
          ### VBD "down" (Calculation start from the beginning of the Voltage array upwards) ###
          for i in range(start_bd_calculation,int(len(Iback))-1):
            if(Vbias[len(Iback)-1]<V_min_kfactor):
                VBD_qa_d = Vbias[len(Iback)-1] 
                break 
      
            if( Iback[i]<I_compliance*(1./current_conversion_value) and Vbias[i]>=V_min_kfactor ):    
              k_qa_d[0] = ( (Iback[i]-Iback[i-1])/(Vbias[i]-Vbias[i-1]) )*(Vbias[i]/Iback[i]) 
              k_qa_d[1] = ( (Iback[i+1]-Iback[i])/(Vbias[i+1]-Vbias[i]) )*(Vbias[i]/Iback[i])
              if( k_qa_d[0]<k_thr and k_qa_d[1]>=k_thr ):
                VBD_qa_d = Vbias[i] 
                break
      
        if(VBD_qa_u!=-1000 and VBD_qa_d!=-1000): 
          vvbd_qa[event.wafer-start_wafer][event.sensor-start_sensor] += VBD_qa_u
          vcount_vbd_qa[event.wafer-start_wafer][event.sensor-start_sensor] += 1
  
        if(I_qa_100V>=0):
          vi_qa[event.wafer-start_wafer][event.sensor-start_sensor] += I_qa_100V
          vcount_i_qa[event.wafer-start_wafer][event.sensor-start_sensor] += 1
          
        #print(str(VBD_qa_u)+"    "+str(I_qa_100V))
        print(str(VBD_qa_u)+"    "+str(VBD_qa_d))
      
        if( VBD_qa_u>(VBD_expected[event.wafer-start_wafer]-3*VBD_expected_sigma[event.wafer-start_wafer]) and VBD_qa_d!=-1000 and VBD_qa_d<=(VBD_expected[event.wafer-start_wafer]-3*VBD_expected_sigma[event.wafer-start_wafer]) ):
          if(verbose): 
            print("!!! SENSOR WITH CURRENT JUMPS ALERT  !!!: from wafer "+str(event.wafer)+" sensor "+str(event.sensor))
        
        if( iv_quality and VBD_qa_u>(VBD_expected[event.wafer-start_wafer]-3*VBD_expected_sigma[event.wafer-start_wafer]) and VBD_qa_u<(VBD_expected[event.wafer-start_wafer]+3*VBD_expected_sigma[event.wafer-start_wafer]) and VBD_qa_d!=-1000 ):
          if( VBD_qa_d>(VBD_expected[event.wafer-start_wafer]-3*VBD_expected_sigma[event.wafer-start_wafer]) ):
           vbump_qa[event.wafer-start_wafer][event.sensor-start_sensor] += 1 
           vcount_bump_qa[event.wafer-start_wafer][event.sensor-start_sensor] += 1 
          else:
           vbump_qa[event.wafer-start_wafer][event.sensor-start_sensor] += 2
           vcount_bump_qa[event.wafer-start_wafer][event.sensor-start_sensor] += 1
        elif( np.logical_not(iv_quality) or (VBD_qa_d!=-1000 and VBD_qa_u!=-1000 and (VBD_qa_u<=(VBD_expected[event.wafer-start_wafer]-3*VBD_expected_sigma[event.wafer-start_wafer]) or VBD_qa_u>=(VBD_expected[event.wafer-start_wafer]+3*VBD_expected_sigma[event.wafer-start_wafer]))) ):
          vbump_qa[event.wafer-start_wafer][event.sensor-start_sensor] += 100
          vcount_bump_qa[event.wafer-start_wafer][event.sensor-start_sensor] += 1
        else: 
          vcount_bump_qa[event.wafer-start_wafer][event.sensor-start_sensor] += -1000
          if(verbose):
            print("Sensor from wafer "+str(event.wafer)+" sensor "+str(event.sensor)+" has Current within acceptance, but VBD could not be calculated. PLEASE CHECK.")
      else:
        vcount_bump_qa[event.wafer-start_wafer][event.sensor-start_sensor] += -1000
        if(verbose):
          print("Sensor from wafer "+str(event.wafer)+" sensor "+str(event.sensor)+" has Voltage and/or Current always equal to zero. PLEASE CHECK.")
    else:
      vcount_bump_qa[event.wafer-start_wafer][event.sensor-start_sensor] += -1000
      if(verbose):
        print("Sensor from wafer "+str(event.wafer)+" sensor "+str(event.sensor)+": Voltage and/or Current vectors have less than "+str(start_bd_calculation)+" elements. PLEASE CHECK.")

222.0    101.99800109863281
!!! SENSOR WITH CURRENT JUMPS ALERT  !!!: from wafer 3 sensor 20
240.01100158691406    102.0
!!! SENSOR WITH CURRENT JUMPS ALERT  !!!: from wafer 3 sensor 17
235.99949645996094    104.0
!!! SENSOR WITH CURRENT JUMPS ALERT  !!!: from wafer 3 sensor 2
234.0    102.00250244140625
!!! SENSOR WITH CURRENT JUMPS ALERT  !!!: from wafer 3 sensor 21
244.0    104.0
!!! SENSOR WITH CURRENT JUMPS ALERT  !!!: from wafer 3 sensor 24
112.0    104.0
226.00750732421875    100.0
!!! SENSOR WITH CURRENT JUMPS ALERT  !!!: from wafer 3 sensor 22
234.0    104.0
!!! SENSOR WITH CURRENT JUMPS ALERT  !!!: from wafer 3 sensor 16
236.0    102.0
!!! SENSOR WITH CURRENT JUMPS ALERT  !!!: from wafer 3 sensor 6
34.01300048828125    34.01300048828125
234.0    102.0
!!! SENSOR WITH CURRENT JUMPS ALERT  !!!: from wafer 3 sensor 11
-1000.0    -1000.0
242.0    102.0
!!! SENSOR WITH CURRENT JUMPS ALERT  !!!: from wafer 3 sensor 15
234.0    100.0
!!! SENSOR WITH CURRENT JUMPS ALERT  !!!: from wa

In [4]:
serial_num_counter=0

mask_sensor = [2,1,6,5,4,3,12,11,10,9,8,7,18,17,16,15,14,13,22,21,20,19,24,23]

for i in range(n_wafer):
    for j in range(n_sensor):
      if(i==2):
          #row = str(wb[(wb['Sensor'] == j+start_sensor)]['Row'].iloc[0])
          #column = str(wb[(wb['Sensor'] == j+start_sensor)]['Column'].iloc[0])
          row = str(wb[(wb['Sensor'] == mask_sensor[j])]['Row'].iloc[0])
          column = str(wb[(wb['Sensor'] == mask_sensor[j])]['Column'].iloc[0])
          row = int(row)
          column = int(column)
          #print(str(row)+"    "+str(column))
    #for j in range(n_col):
        #for k in range(n_row):
          #if( i==0 ):
          if( i<9 ):
             if(vcount_bump_qa[i][mask_sensor[j]-1] != 0):
               #serial_num = str(wb[(wb['Wafer'] == i+start_wafer) & (wb['Row'] == k+start_row) & (wb['Column'] == j+start_col)]['SerialNumber'].iloc[0])
               #serial_num = str(wb[(wb['Sensor'] == j+start_sensor)]['SerialNumber'].iloc[0])
               serial_num = str(wb[(wb['Sensor'] == mask_sensor[j])]['SerialNumber'].iloc[0])
               
               
             if(vcount_vbd_qa[i][mask_sensor[j]-1]!=0): 
               #hV_qa[i].Fill( j+start_col, k+start_row, vvbd_qa[i][j][k]/vcount_vbd_qa[i][j][k] )
               print(str(row)+"    "+str(column))
               hV_qa[i].Fill( column, row, vvbd_qa[i][mask_sensor[j]-1]/vcount_vbd_qa[i][mask_sensor[j]-1] )
               #hV_qa[i].SetBinContent(hV_qa[i].FindBin(column, row), vvbd_qa[i][mask_sensor[j]-1]/vcount_vbd_qa[i][mask_sensor[j]-1] )
          
               
             if(vcount_i_qa[i][mask_sensor[j]-1]!=0): 
               #hI_qa_100V[i].Fill( j+start_col, k+start_row, vi_qa[i][j][k]/vcount_i_qa[i][j][k] )
               hI_qa_100V[i].Fill( column, row, vi_qa[i][mask_sensor[j]-1]/vcount_i_qa[i][mask_sensor[j]-1] )
             
  
             if(vcount_bump_qa[i][mask_sensor[j]-1]<0): 
               #hnoisy_qa[i].Fill(j+start_col,k+start_row,-1000)
               
               pippo = {'component': serial_num,
                        'type': 'Sensor IV - ETL Site',
                        'measurement_date': dtz.isoformat(), #year, month, day, hour, minute, second
                        'location': 'Universita e INFN Torino',
                        'user_created': 'fsiviero',
                        'version': '0.0',
                        'data':{
                         'leakage_current_uA': None,
                         'breakdown_voltage_V': None, 
                         'category': None,
                         'gain_category': None,  
                         'current': vi_raw[i][mask_sensor[j]-1],
                         'voltage': v_raw[i][mask_sensor[j]-1]
                        }
                       }
               
               tests_json.append(pippo)
          
             if( vcount_bump_qa[i][mask_sensor[j]-1] > 0 ):
               total_sensors_counter_qa+=1
               
               vendor_leakage_json = str()
               vendor_vbd_json = str()
               vendor_category_json = str()
               
               if(vcount_i_qa[i][mask_sensor[j]-1]!=0): 
                 vendor_leakage_json = float(vi_qa[i][mask_sensor[j]-1]/vcount_i_qa[i][mask_sensor[j]-1])
               else:
                 vendor_leakage_json = None
                 
               if(vcount_vbd_qa[i][mask_sensor[j]-1]!=0):
                 vendor_vbd_json = float(vvbd_qa[i][mask_sensor[j]-1]/vcount_vbd_qa[i][mask_sensor[j]-1])
               else:
                 vendor_vbd_json = None
                 
               if( vbump_qa[i][mask_sensor[j]-1]/vcount_bump_qa[i][mask_sensor[j]-1]==1 ):
                 hbump_qa[i].Fill(column, row, 1)
                 #hnoisy_qa[i].Fill(j+start_col,k+start_row,0)
                 good_sensors_counter_qa+=1
                 vendor_category_json = "GOOD"
               elif( vbump_qa[i][mask_sensor[j]-1]/vcount_bump_qa[i][mask_sensor[j]-1]>2 ):
                 hbump_qa[i].Fill(column, row, 3)
                 #hnoisy_qa[i].Fill(j+start_col,k+start_row,0)
                 bad_sensors_counter_qa+=1
                 vendor_category_json = "BAD"
               elif( vbump_qa[i][mask_sensor[j]-1]/vcount_bump_qa[i][mask_sensor[j]-1]>1 and vbump_qa[i][mask_sensor[j]-1]/vcount_bump_qa[i][mask_sensor[j]-1]<=2 ):
                 #hbump_qa[i].Fill(column, row, 2) #!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
                 hbump_qa[i].Fill(column, row, 1)
                 #hnoisy_qa[i].Fill(j+start_col,k+start_row,1)
                 medium_sensors_counter_qa+=1
                 #vendor_category_json = "MEDIUM" !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
                 vendor_category_json = "GOOD"

               if vendor_category_json is "BAD":
                 vendor_gain_category_json=None
               else:
                if vendor_vbd_json is None:
                 vendor_gain_category_json=None
                elif( vendor_vbd_json>(VBD_expected[i]-VBD_expected_sigma[i]/2.) and vendor_vbd_json<(VBD_expected[i]+VBD_expected_sigma[i]/2.) ):
                 vendor_gain_category_json='B'
                elif( vendor_vbd_json<=(VBD_expected[i]-VBD_expected_sigma[i]/2.) and vendor_vbd_json>(VBD_expected[i]-3*VBD_expected_sigma[i]) ):
                 vendor_gain_category_json='A' 
                elif( vendor_vbd_json>=(VBD_expected[i]+VBD_expected_sigma[i]/2.) and vendor_vbd_json<(VBD_expected[i]+3*VBD_expected_sigma[i]) ):
                 vendor_gain_category_json='C' 
               
               pippo = {'component': serial_num,
                        'type': 'Sensor IV - ETL Site',
                        'measurement_date': dtz.isoformat(), #year, month, day, hour, minute, second
                        'location': 'Universita e INFN Torino',
                        'user_created': 'fsiviero',
                        'version': '0.0',
                        'data':{
                         'leakage_current_uA': vendor_leakage_json,
                         'breakdown_voltage_V': vendor_vbd_json, 
                         'category': vendor_category_json,  
                         'gain_category': vendor_gain_category_json,
                         'current': vi_raw[i][mask_sensor[j]-1],
                         'voltage': v_raw[i][mask_sensor[j]-1]
                        }
                       }
               
               tests_json.append(pippo)

if write_json:
 with open(save_path+json_filename+".json", 'w') as f:
   json.dump(tests_json, f, indent=4)

if(verbose):
  print("Fraction of GOOD sensors (on-wafer): "+str(good_sensors_counter_qa/total_sensors_counter_qa))
  print("Fraction of MEDIUM sensors (on-wafer): "+str(medium_sensors_counter_qa/total_sensors_counter_qa))
  print("Fraction of BAD sensors (on-wafer): "+str(bad_sensors_counter_qa/total_sensors_counter_qa))
  print("Fraction of non-empty IVs: "+str(non_empty_counter/dumb_counter))
  print("Fraction of IVs with non-zero values: "+str(non_zero_counter/dumb_counter))

1    3
2    2
2    3
3    1
3    2
4    1
4    2
4    3
4    4
5    2
5    3
5    4
6    3
6    4
Fraction of GOOD sensors (on-wafer): 0.0
Fraction of MEDIUM sensors (on-wafer): 0.8
Fraction of BAD sensors (on-wafer): 0.2
Fraction of non-empty IVs: 1.0
Fraction of IVs with non-zero values: 1.0


<>:93: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:93: SyntaxWarning: "is" with a literal. Did you mean "=="?
/var/folders/qv/rs5lq5zd0kzd81_zslwl_h880000gn/T/ipykernel_66526/3173754769.py:93: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if vendor_category_json is "BAD":


In [ ]:
cI_100V = []
cV = []
cbump = []
cnoisy = []

for i in range(n_wafer):
  if(bcurrent):
    cI_100V.append(root.TCanvas("c_I_"+str(V_current_level[i])+"V_W"+str(i+1), "c I@"+str(V_current_level[i])+"V_W"+str(i+1), 1000,1000))
  if(bvoltage):
    cV.append(root.TCanvas("c_V_W"+str(i+1), "c VBD_W"+str(i+1), 1000,1000))
  if(bcategory):
    cbump.append(root.TCanvas("c_bump_W"+str(i+1), "c bump_W"+str(i+1), 1000,1000))
  if(bnoisy):
    cnoisy.append(root.TCanvas("c_noisy_W"+str(i+1), "c noisy_W"+str(i+1), 1000,1000))


for i in range(n_wafer):
 if( i==2 ):
 #if( i<9 ):
    #hI_qa_100V[i]->GetZaxis()->SetRangeUser( 0.1, hI_qa_100V[i]->GetMaximum() ) #Alternative colored axis range
    hI_qa_100V[i].GetZaxis().SetRangeUser( low_iv_range, high_iv_range )
    hI_qa_100V[i].GetZaxis().SetTitle("[nA]")
    hI_qa_100V[i].GetXaxis().SetTitle("column")
    hI_qa_100V[i].GetYaxis().SetTitle("row")
    hI_qa_100V[i].SetMarkerSize(3.)

  
    #hV_qa[i]->GetZaxis()->SetRangeUser( hV_qa[i]->GetMinimum(), hV_qa[i]->GetMaximum() ); //Alternative colored axis range
    hV_qa[i].GetZaxis().SetRangeUser( low_vbd_range, high_vbd_range )
    hV_qa[i].GetZaxis().SetTitle("[V]")
    hV_qa[i].GetXaxis().SetTitle("column") 
    hV_qa[i].GetYaxis().SetTitle("row")
    hV_qa[i].SetMarkerSize(3.)

    hbump_qa[i].GetZaxis().SetRangeUser( 1, 3 )
    hbump_qa[i].GetZaxis().SetTitle("Category")
    hbump_qa[i].GetXaxis().SetTitle("column")
    hbump_qa[i].GetYaxis().SetTitle("row")

    #hnoisy_qa[i].GetZaxis().SetRangeUser( -0.1, 1 )
    #hnoisy_qa[i].GetZaxis().SetTitle("Presence of bad pad(s)")
    #hnoisy_qa[i].GetXaxis().SetTitle("column")
    #hnoisy_qa[i].GetYaxis().SetTitle("row")


    if(bcurrent):
      #cI_100V.append(root.TCanvas("c_I_"+str(V_current_level)+"V_W"+str(i+1), "c I@"+str(V_current_level)+"V_W"+str(i+1), 1000,1000))
      cI_100V[i].SetRightMargin(0.15)
      cI_100V[i].cd()
      hI_qa_100V[i].SetMarkerSize(0.9)
      hI_qa_100V[i].Draw("textcolz")
      root.gPad.SetGrid(1,1)
      root.gPad.SetLogz(1)
      root.gPad.Update()
      cI_100V[i].Update()
      hI_qa_100V[i].GetXaxis().SetNdivisions(n_col)
      hI_qa_100V[i].GetYaxis().SetNdivisions(n_row)
  
      if(save):
        #cI_100V[i].SaveAs("pics/I_"+str(V_current_level)+"V_W"+str(i+1)+"_FINAL_py.png")
        cI_100V[i].SaveAs(save_path+"I_"+str(V_current_level[i])+"V_W"+str(i+1)+".png")


    if(bvoltage):
      root.gStyle.SetPaintTextFormat("1.1f")
      #cV.append(root.TCanvas("c_V_W"+str(i+1), "c VBD_W"+str(i+1), 1000,1000))
      cV[i].SetRightMargin(0.15)
      cV[i].cd()
      hV_qa[i].SetMarkerSize(0.9)
      hV_qa[i].Draw("textcolz")
      #hV_qa[i].SaveAs(save_path+"VBD_W"+str(i+1)+".root")
      root.gPad.SetGrid(1,1)
      root.gPad.Update()
      cV[i].Update()
      hV_qa[i].GetXaxis().SetNdivisions(n_col)
      hV_qa[i].GetYaxis().SetNdivisions(n_row)
  
      if(save): 
        #cV[i].SaveAs( "pics/VBD_W"+str(i+1)+"_FINAL_py.png",i+1)
        cV[i].SaveAs(save_path+"VBD_W"+str(i+1)+".png")


    if(bcategory):
      #cbump.append(root.TCanvas("c_bump_W"+str(i+1), "c bump_W"+str(i+1), 1000,1000))
      cbump[i].SetRightMargin(0.15)
      cbump[i].cd()
      hbump_qa[i].Draw("colz")
      root.gPad.SetGrid(1,1)
      root.gPad.Update()
      cbump[i].Update()
      hbump_qa[i].GetXaxis().SetNdivisions(n_col)
      hbump_qa[i].GetYaxis().SetNdivisions(n_row)
       
      if(save):
        #cbump[i].SaveAs("pics/categories_W"+str(i+1)+"_FINAL_py.png" )
        cbump[i].SaveAs(save_path+"categories_W"+str(i+1)+".png" )   

'''
    if(bnoisy):
      #cnoisy.append(root.TCanvas("c_noisy_W"+str(i+1), "c noisy_W"+str(i+1), 1000,1000))
      cnoisy[i].SetRightMargin(0.15)
      cnoisy[i].cd()
      hnoisy_qa[i].Draw("colz")
      root.gPad.SetGrid(1,1)
      root.gPad.Update()
      cnoisy[i].Update()
      hnoisy_qa[i].GetXaxis().SetNdivisions(n_col)
      hnoisy_qa[i].GetYaxis().SetNdivisions(n_row)

      if(save):
        cnoisy[i].SaveAs("pics/noisy_W"+str(i+1)+".png")
'''